# 02 - Scanned PDF NER (Task 3)
Pipeline: scanned PDF → PyMuPDF + Tesseract OCR (Spanish) → extracted text → GLiNER (small + merge)
→ strict/relaxed evaluation. Only 10 scanned PDFs are available (test-0001 … test-0010).

In [ ]:
import os, sys, warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"                 # evitar la descarga que se colgaba
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"    # ocultar aviso de symlinks
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))          # para poder importar 'ner'

%load_ext autoreload
%autoreload 2

In [ ]:
import difflib
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed, ENTITY_LABELS
from ner.inference import load_gliner_model, predict_dataset, merge_dataset
from ner.pdf import extract_text_from_pdf, extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_scanned'
LABELS = list(ENTITY_LABELS)

gold_samples = load_jsonl(VALIDATION_PATH)
print(len(gold_samples), 'gold documents')

In [ ]:
pdf_path = f'{PDF_DIR}/test-0001_scanned.pdf'

no_ocr = extract_text_from_pdf(pdf_path, ocr=False)     # solo la capa de texto (vacía)
with_ocr = extract_text_from_pdf(pdf_path, ocr=True)    # Tesseract sobre la imagen

print('Characters WITHOUT OCR:', len(no_ocr))
print('Characters WITH OCR   :', len(with_ocr), '| gold:', len(gold_samples[0]['text']))
print('\n', with_ocr[:400])

In [ ]:
scanned = extract_pdf_dataset(PDF_DIR, gold_samples, mode='blocks', ocr=True)
print(len(scanned), 'scanned PDFs extracted\n')

for doc, gold in zip(scanned, gold_samples):
    matcher = difflib.SequenceMatcher(None, gold['text'], doc['text'])
    # Fragmentos distintos: (lo que pone el gold, lo que ha leído el OCR)
    diffs = [(gold['text'][a:b], doc['text'][c:d])
             for tag, a, b, c, d in matcher.get_opcodes() if tag != 'equal']
    print(f"{doc['id']}  similarity={matcher.ratio():.4f}  identical={doc['text'] == gold['text']}  "
          f"differences (gold -> OCR): {diffs[:5]}")

In [ ]:
model = load_gliner_model()

OUT_PATH = '../results/pdf_scanned/gliner_small_merge_ocr.jsonl'
preds = predict_dataset(model, scanned, LABELS, threshold=0.5)
save_responses_to_jsonl(merge_dataset(preds), OUT_PATH)

strict = evaluate_ner_strict(VALIDATION_PATH, OUT_PATH)
relaxed = evaluate_ner_relaxed(VALIDATION_PATH, OUT_PATH)

In [ ]:
plain_preds = load_jsonl('../results/plain/gliner_small_merge.jsonl', n=10)
PLAIN10_PATH = '../results/pdf_scanned/plain_first10_reference.jsonl'
save_responses_to_jsonl(plain_preds, PLAIN10_PATH)

print('########## PLAIN TEXT (same 10 documents) ##########')
evaluate_ner_strict(VALIDATION_PATH, PLAIN10_PATH)
evaluate_ner_relaxed(VALIDATION_PATH, PLAIN10_PATH)

### Task 3 — Results
...